# Baselines: is fine-tuning worth it?

The adapter sounds like me (StyleClassifier.ipynb) but barely answers the conversation (Relevance.ipynb). Before spending more on fine-tuning, compare it with what you'd try without training, on exactly the same evaluation (the 1,133 test conversations, the same frozen style classifiers, the same relevance measure):

1. **Retrieval:** for each test conversation, my real reply to the most similar training conversation. No model.
2. **Few-shot prompting:** Qwen2.5 0.5B, 1.5B and 3B without training, with 20 real replies of mine from train in the system prompt.
3. **The same recipe on a bigger model:** LoRA on Qwen2.5-1.5B, checkpoint chosen on validation as in decision 15.

Relevance is re-scored for every set with Qwen2.5-7B, because the 1.5B, the judge of decision 20, now generates replies here.

## Prediction (written before running)

From what we've seen, a more powerful model should improve what gets said: the replies should make more sense in the conversation (higher relevance) while keeping my style. Style alone isn't the question, since we already know a very small model can get it. But it has to be measured too: the bigger model has to keep my style, not just answer better.

In [1]:
# Each heavy step runs a script from scripts/ through run(), which stops the notebook on failure. They all resume: re-running this notebook skips
# what is already done (trainings with a final adapters.safetensors, generations already written,
# evaluations already in their JSON). Run headless with:
#   cd experiments && nohup uv run jupyter nbconvert --to notebook --execute --inplace \
#       --ExecutePreprocessor.timeout=-1 Baselines.ipynb > Baselines.log 2>&1 &
import json, os, subprocess
from pathlib import Path
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
ROOT = Path("..").resolve()
R = ROOT / "experiments" / "results"
M05 = "mlx-community/Qwen2.5-0.5B-Instruct-4bit"
M15 = "mlx-community/Qwen2.5-1.5B-Instruct-4bit"
M3 = "mlx-community/Qwen2.5-3B-Instruct-4bit"
JUDGE = "mlx-community/Qwen2.5-7B-Instruct-4bit"  # generates and trains nothing here
V6 = "data/dataset/session/valid.jsonl"
V20 = "data/dataset/context/valid_ctx20.jsonl"
TURNS_V1 = "data/processed_v1_2026-10-17/turns.jsonl"  # the turns session/ was built from

def run(cmd):
    """Run a command from the repo root, show its output here, and stop the notebook if it fails."""
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if "Fetching" not in line:
            print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"failed ({p.returncode}): {cmd}")

def done_training(name):
    return (ROOT / "adapters" / name / "adapters.safetensors").exists()

def best_step(path):
    return json.load(open(path))["best"]["step"]

## 1. Retrieval

In [2]:
run("uv run python scripts/baseline_retrieval.py")

$ uv run python scripts/baseline_retrieval.py
1133 replies retrieved from 5404 train examples -> data/style/generations_retrieval.jsonl


## 2. Few-shot prompting (temperature 1.0, 20 examples of mine in the system prompt)

In [3]:
for tag, model in [("0.5b", M05), ("1.5b", M15), ("3b", M3)]:
    print("few-shot", tag)
    run(f"uv run python scripts/generate.py --name fewshot20_{tag} --model {model} --few-shot 20")

few-shot 0.5b
$ uv run python scripts/generate.py --name fewshot20_0.5b --model mlx-community/Qwen2.5-0.5B-Instruct-4bit --few-shot 20
400/1133 (1.1 min)
500/1133 (2.5 min)
600/1133 (3.7 min)
700/1133 (5.0 min)
800/1133 (6.2 min)
900/1133 (7.7 min)
1000/1133 (9.0 min)
1100/1133 (10.3 min)
1133 replies in data/style/generations_fewshot20_0.5b.jsonl
few-shot 1.5b
$ uv run python scripts/generate.py --name fewshot20_1.5b --model mlx-community/Qwen2.5-1.5B-Instruct-4bit --few-shot 20
100/1133 (1.8 min)
200/1133 (3.4 min)
300/1133 (4.9 min)
400/1133 (6.3 min)
500/1133 (7.7 min)
600/1133 (9.4 min)
700/1133 (10.7 min)
800/1133 (12.4 min)
900/1133 (14.1 min)
1000/1133 (15.6 min)
1100/1133 (17.4 min)
1133 replies in data/style/generations_fewshot20_1.5b.jsonl
few-shot 3b
$ uv run python scripts/generate.py --name fewshot20_3b --model mlx-community/Qwen2.5-3B-Instruct-4bit --few-shot 20
100/1133 (3.3 min)
200/1133 (6.4 min)
300/1133 (9.3 min)
400/1133 (11.9 min)
500/1133 (14.7 min)
600/1133 (17.

## 3. The same recipe on Qwen2.5-1.5B

In [4]:
if not done_training("run1_1.5b"):
    run(f"uv run python scripts/train.py --name run1_1.5b --data data/dataset/session --model {M15} --iters 3500")

$ uv run python scripts/train.py --name run1_1.5b --data data/dataset/session --model mlx-community/Qwen2.5-1.5B-Instruct-4bit --iters 3500
5404 train examples, batch 4: 3500 steps (2.59 epochs)
.venv/bin/python3 -m mlx_lm lora --model mlx-community/Qwen2.5-1.5B-Instruct-4bit --train --data data/dataset/session --mask-prompt --adapter-path adapters/run1_1.5b --iters 3500 --batch-size 4 --learning-rate 1e-05 --num-layers 16 --max-seq-length 2048 --save-every 100 --steps-per-eval 100 --val-batches 25 --seed 0
Loading pretrained model
Loading datasets
Training
╭─ mlx_lm.lora ────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│    model           mlx-community/Qwen2.5-1.5B-Instruct-4bit                                                          │
│    type            lora · 16 layers · rank 8                                                                         │
│    dataset         data/dataset/session                                

In [5]:
run(f"uv run python scripts/eval_checkpoints.py adapters/run1_1.5b --model {M15} --data {V6} --steps 500:3500:500 --out experiments/results/checkpoints_run1_1.5b.json")
S15 = best_step(R / "checkpoints_run1_1.5b.json")
print("best step", S15)

$ uv run python scripts/eval_checkpoints.py adapters/run1_1.5b --model mlx-community/Qwen2.5-1.5B-Instruct-4bit --data data/dataset/session/valid.jsonl --steps 500:3500:500 --out experiments/results/checkpoints_run1_1.5b.json
step   500: loss 3.220
step  1000: loss 3.168
step  1500: loss 3.156
step  2000: loss 3.165
step  2500: loss 3.150
step  3000: loss 3.206
step  3500: loss 3.212
best: step 2500, loss 3.150
best step 2500


In [6]:
run(f"uv run python scripts/generate.py --name adapter1.5b_t1 --model {M15} --adapter adapters/run1_1.5b --step {S15}")
run(f"uv run python scripts/eval_signal.py --adapter adapters/run1_1.5b --step {S15} --model {M15} --turns {TURNS_V1} --out experiments/results/signal_run1_1.5b.json")

$ uv run python scripts/generate.py --name adapter1.5b_t1 --model mlx-community/Qwen2.5-1.5B-Instruct-4bit --adapter adapters/run1_1.5b --step 2500
100/1133 (1.2 min)
200/1133 (2.3 min)
300/1133 (3.2 min)
400/1133 (4.1 min)
500/1133 (5.0 min)
600/1133 (6.3 min)
700/1133 (7.3 min)
800/1133 (8.3 min)
900/1133 (9.6 min)
1000/1133 (10.8 min)
1100/1133 (11.8 min)
1133 replies in data/style/generations_adapter1.5b_t1.jsonl
$ uv run python scripts/eval_signal.py --adapter adapters/run1_1.5b --step 2500 --model mlx-community/Qwen2.5-1.5B-Instruct-4bit --turns data/processed_v1_2026-10-17/turns.jsonl --out experiments/results/signal_run1_1.5b.json
mine:    base 60.9 -> adapter 23.0 (62.3% drop)
friends: base 58.5 -> adapter 35.0 (40.1% drop)
signal: 22.2 points, 95% CI [16.3, 29.1]


## 4. Style: frozen classifiers on every set

In [7]:
EXTRA = "retrieval fewshot20_0.5b fewshot20_1.5b fewshot20_3b adapter1.5b_t1"
run(f"uv run python scripts/eval_style.py --adapter-gens adapter_t1 --base-gens base_t1 --extra {EXTRA} --transformer --out experiments/results/style_baselines.json --force")

$ uv run python scripts/eval_style.py --adapter-gens adapter_t1 --base-gens base_t1 --extra retrieval fewshot20_0.5b fewshot20_1.5b fewshot20_3b adapter1.5b_t1 --transformer --out experiments/results/style_baselines.json --force
.venv/lib/python3.13/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
length (test AUC on real messages 0.551)
  base       classified as me   7.8% [5.6%, 9.5%]
  adapter    classified as me  41.9% [39.6%, 44.4%]
  retrieval  classified as me  51.7% [47.7%, 56.3%]
  fewshot20_0.5b classified as me   9.3% [6.7%, 11.1%]
  fewshot20_1.5b classified as me  11.8% [9.2%, 14.0%]
  fewshot20_3b classified as me  11.6% [9.0%, 13.2%]
  adapter1.5b_t1 classified as me  39.1% [35.6%, 42.9%]
  real (me)  classified as me  47.4% [43.1%, 51.4%]
  friends    classified as me  38.7% [32.8%, 44.8%]
  position of adapter: 0.86 [0.77, 0.97]
  position of retrieval: 1.11 [0

## 4b. Retrieval from the validation pool

The style classifiers were trained on the train replies, so copying a train reply scores as "me" by construction (1.37 / 1.38 above). Copying from the validation pool, which the classifiers never saw, gives the honest style number.

In [ ]:
run("uv run python scripts/baseline_retrieval.py --pool valid")
if not (R / "style_retrieval_valid.json").exists():
    run("uv run python scripts/eval_style.py --adapter-gens adapter_t1 --base-gens base_t1 --extra retrieval_valid --transformer --out experiments/results/style_retrieval_valid.json")

## 5. Relevance, judged by Qwen2.5-7B

In [8]:
run(f"uv run python scripts/eval_relevance.py --adapter-gens adapter_t1 --base-gens base_t1 --extra {EXTRA} retrieval_valid --scorer {JUDGE} --cache experiments/results/relevance_nll_7b.json --out experiments/results/relevance_7b.json --force")

$ uv run python scripts/eval_relevance.py --adapter-gens adapter_t1 --base-gens base_t1 --extra retrieval fewshot20_0.5b fewshot20_1.5b fewshot20_3b adapter1.5b_t1 --scorer mlx-community/Qwen2.5-7B-Instruct-4bit --cache experiments/results/relevance_nll_7b.json --out experiments/results/relevance_7b.json --force
floor (other conv.): 100/1133 (1.8 min)
floor (other conv.): 200/1133 (4.0 min)
floor (other conv.): 300/1133 (6.0 min)
floor (other conv.): 400/1133 (7.4 min)
floor (other conv.): 500/1133 (9.0 min)
floor (other conv.): 600/1133 (10.6 min)
floor (other conv.): 700/1133 (12.0 min)
floor (other conv.): 800/1133 (13.6 min)
floor (other conv.): 900/1133 (15.3 min)
floor (other conv.): 1000/1133 (16.9 min)
floor (other conv.): 1100/1133 (18.7 min)
floor (other conv.): 1133/1133 (19.2 min)
base t1.0: 100/1133 (21.7 min)
base t1.0: 200/1133 (23.8 min)
base t1.0: 300/1133 (26.1 min)
base t1.0: 400/1133 (28.2 min)
base t1.0: 500/1133 (30.3 min)
base t1.0: 600/1133 (32.6 min)
base t1.0:

## Summary

In [9]:
style = json.load(open(R / "style_baselines.json"))["scores"]
style_valid = json.load(open(R / "style_retrieval_valid.json"))["scores"]
for m in style:
    style[m]["position retrieval_valid"] = style_valid[m]["position retrieval_valid"]
rel = json.load(open(R / "relevance_7b.json"))
names = {"adapter": "fine-tuned 0.5B (ours)", "adapter t1.0": "fine-tuned 0.5B (ours)", "adapter1.5b_t1": "fine-tuned 1.5B",
         "retrieval": "retrieval (train pool)", "retrieval_valid": "retrieval (valid pool)", "fewshot20_0.5b": "few-shot 0.5B", "fewshot20_1.5b": "few-shot 1.5B", "fewshot20_3b": "few-shot 3B"}
rows = ["adapter", "adapter1.5b_t1", "fewshot20_0.5b", "fewshot20_1.5b", "fewshot20_3b", "retrieval", "retrieval_valid"]
print(f"{'set':<24} {'style TF-IDF':>14} {'style mDeBERTa':>15} {'relevance (7B)':>16}")
for k in rows:
    t = style["tfidf"][f"position {k}"]; d = style["transformer"][f"position {k}"]
    rk = "adapter t1.0" if k == "adapter" else k
    r = rel["positions"][rk]
    print(f"{names[k]:<24} {t['value']:>6.2f} [{t['ci'][0]:.2f},{t['ci'][1]:.2f}] {d['value']:>6.2f} [{d['ci'][0]:.2f},{d['ci'][1]:.2f}] {r['value']:>6.2f} [{r['ci'][0]:.2f},{r['ci'][1]:.2f}]")
print("style position: 0 = base model, 1 = me | relevance position: 0 = my reply to another conversation, 1 = my real reply")

set                        style TF-IDF  style mDeBERTa   relevance (7B)
fine-tuned 0.5B (ours)     0.96 [0.84,1.08]   0.81 [0.69,0.90]   0.27 [0.09,0.42]
fine-tuned 1.5B            1.08 [0.91,1.28]   0.94 [0.86,1.03]   0.33 [0.19,0.54]
few-shot 0.5B             -0.02 [-0.16,0.07]  -0.25 [-0.45,-0.09]  -0.55 [-0.76,-0.38]
few-shot 1.5B              0.47 [0.32,0.59]   0.32 [0.16,0.42]  -0.11 [-0.29,0.03]
few-shot 3B                0.32 [0.16,0.41]   0.59 [0.46,0.68]   0.27 [0.07,0.46]
retrieval (train pool)     1.37 [1.18,1.58]   1.38 [1.21,1.60]   0.40 [0.22,0.58]
retrieval (valid pool)     1.05 [0.95,1.14]   1.09 [0.98,1.19]   0.37 [0.14,0.64]
style position: 0 = base model, 1 = me | relevance position: 0 = my reply to another conversation, 1 = my real reply


## 6. Paired differences in relevance

The intervals above are per set. Two sets scored on the same contexts can be compared directly: the difference in context gain, with the same bootstrap samples of sessions. This is the test for "retrieval answers better than the adapter".

In [ ]:
# Paired difference in relevance gain between sets: same contexts, same bootstrap samples of sessions.
import json
from collections import defaultdict
import numpy as np
from talk_like_me.config import DATA
from talk_like_me.io import read_jsonl
from talk_like_me.stats import interval, resample

nll = json.load(open(R / "relevance_nll_7b.json"))
sessions = [r["session"] for r in read_jsonl(DATA / "style" / "test.jsonl") if r["label"] == 1]
sums = {}
for name, rows in nll.items():
    s_ = defaultdict(lambda: np.zeros(3))
    for (a, n, b, _), s in zip(rows, sessions):
        s_[s] += (a, b, n)
    sums[name] = dict(s_)

def gain(name, chosen):
    tot = sum(sums[name][s] for s in chosen)
    return (tot[1] - tot[0]) / tot[2]

pairs = [("retrieval", "adapter t1.0"), ("retrieval", "adapter1.5b_t1"), ("adapter1.5b_t1", "adapter t1.0"),
         ("fewshot20_3b", "adapter t1.0"), ("adapter t1.0", "floor (other conv.)"), ("retrieval_valid", "adapter t1.0"), ("retrieval_valid", "retrieval")]
samples = resample(set(sessions))
allS = sorted(set(sessions))
for a, b in pairs:
    d = gain(a, allS) - gain(b, allS)
    boot = [gain(a, c) - gain(b, c) for c in samples]
    lo, hi = interval(boot)
    print(f"{a:<16} - {b:<20} {d:+.3f} nats [{lo:+.3f}, {hi:+.3f}]  P(diff>0) = {np.mean(np.array(boot) > 0):.2f}")


retrieval        - adapter t1.0         +0.054 nats [-0.022, +0.163]  P(diff>0) = 0.92
retrieval        - adapter1.5b_t1       +0.028 nats [-0.054, +0.098]  P(diff>0) = 0.76
adapter1.5b_t1   - adapter t1.0         +0.026 nats [-0.050, +0.157]  P(diff>0) = 0.77
fewshot20_3b     - adapter t1.0         -0.001 nats [-0.077, +0.104]  P(diff>0) = 0.52
adapter t1.0     - floor (other conv.)  +0.122 nats [+0.036, +0.188]  P(diff>0) = 1.00
retrieval_valid  - adapter t1.0         +0.044 nats [-0.037, +0.170]  P(diff>0) = 0.84
retrieval_valid  - retrieval            -0.010 nats [-0.110, +0.096]  P(diff>0) = 0.44


## Conclusion

**Prediction:** a bigger model improves what gets said (relevance) while keeping my style.

**Half right.** The 1.5B adapter keeps my style best of all the generated sets (style position 1.08 TF-IDF, 0.94 mDeBERTa) and its relevance is a bit higher than the 0.5B one (0.33 vs 0.27), but the paired difference is +0.026 nats [-0.050, +0.157]: no demonstrable improvement in what it says.

**Style: fine-tuning wins clearly.** No prompt gets close, not even with a 3B model and 20 examples of mine (0.32 / 0.59). Few-shot 0.5B doesn't move from the base model (-0.02 / -0.25).

**Relevance: retrieval looks best at first, but it isn't demonstrably better.** Copying one of my real replies from a similar conversation has the highest mean (position 0.40 [0.22, 0.58]), above both adapters. But paired with the same contexts, retrieval minus the 0.5B adapter is +0.054 nats [-0.022, +0.163], and minus the 1.5B adapter +0.028 [-0.054, +0.098]. The intervals overlap almost completely: the adapter's mean is shifted down, the noise is the same. Few-shot 3B ties the 0.5B adapter (-0.001 nats). The 0.5B adapter does beat the floor (+0.122 [+0.036, +0.188]), so it uses the conversation, just not much.

**The judge mattered.** With Qwen2.5-7B the band between the floor and my real reply is 0.44 nats (0.214 to 0.658) instead of 0.24 with the 1.5B judge, and the 0.5B adapter's relevance no longer includes 0 (0.27 [0.09, 0.42]; it was 0.20 [-0.11, 0.45]).

**Limits.**
- The train-pool retrieval *style* numbers (1.37 / 1.38) are inflated: the classifiers were trained on those same replies. From the validation pool, which they never saw, style is 1.05 / 1.09: as "me" as my real replies, which is what they are. Its relevance is the same as the train pool's (0.37 [0.14, 0.64]; paired -0.010 nats [-0.110, +0.096]), so the relevance result above doesn't depend on that leak.
- The judge has the same information as the generator: it can't know the facts of my life, so it can't reward a reply for being *true*, only for fitting the conversation (how much the context makes the reply predictable). It still separates my real replies from the floor by 0.44 nats, so it measures something real, but "relevance" here means coherence with the conversation, not right content.

**What it suggests (not tested):** the adapter gives the form and retrieval comparable content. Adapter + similar replies of mine in the context is the natural next experiment; it's a new front.

> **Tested on 7 Nov in AdapterRetrieval.ipynb:** with my replies to the 3 most similar validation conversations in its prompt, the adapter's relevance is 0.34 (paired +0.027 nats against the adapter alone, CI -0.040 to +0.109): not demonstrably better.
